# 00 — DLC SuperAnimal-Quadruped smoke test (sample horse video)

**Cel:** sprawdzić, czy DLC 3.x z modelem SuperAnimal-Quadruped uruchamia się na lokalnym Macu i nakłada keypoints wzrokowo sensownie na sample horse clip.

**Wynik:** annotated MP4 w `../outputs/sample_with_keypoints.mp4` + 5 podglądowych klatek.

**Gate (zob. `../GATE.md`):** punkty 1 (setup) i 2 (sensowne keypoints).

**Wymagania:** uruchomione `bash setup.sh` z folderu `poc/`.

## 1. Sanity check — wersje, weights, sample video

In [ ]:
import os, sys, glob, platform
from pathlib import Path

POC = Path(os.getcwd()).resolve()
if POC.name == 'notebooks':
    POC = POC.parent
os.chdir(POC)
print('POC dir:', POC)
print('Python :', sys.version.split()[0])
print('Platform:', platform.platform())

import torch
print('Torch  :', torch.__version__, '| MPS dostepny:', torch.backends.mps.is_available())

import deeplabcut
print('DLC    :', deeplabcut.__version__)

sample = POC / 'data' / 'sample_horse.mp4'
weights_dir = POC / 'checkpoints' / 'superanimal-quadruped'
assert sample.exists(), f'BRAK sample video w {sample}. Uruchom bash setup.sh'
print('Sample :', sample, f'({sample.stat().st_size / 1e6:.1f} MB)')
print('Weights:', weights_dir, '(istnieje)' if weights_dir.exists() else '(BRAK — DLC pobierze lazy)')

## 2. Inferencja SuperAnimal-Quadruped (zero-shot)

DLC 3.x ma `video_inference_superanimal` które od razu pobiera weights (jeśli brak), nakłada keypoints, zapisuje annotated MP4 i tabelę współrzędnych (.h5).

In [ ]:
out_dir = POC / 'outputs'
out_dir.mkdir(exist_ok=True)

deeplabcut.video_inference_superanimal(
    videos=[str(sample)],
    superanimal_name='superanimal_quadruped',
    model_name='hrnet_w32',
    detector_name='fasterrcnn_resnet50_fpn_v2',
    video_adapt=False,
    dest_folder=str(out_dir),
    pseudo_threshold=0.1,
)

## 3. Wzrokowa inspekcja — 5 klatek z overlayem

In [ ]:
import cv2, matplotlib.pyplot as plt, numpy as np

annotated = sorted(out_dir.glob('*labeled.mp4'))
if not annotated:
    annotated = sorted(out_dir.glob('*_labeled*.mp4'))
assert annotated, f'Nie znaleziono annotated video w {out_dir}. Sprawdz logi powyzej.'
vid_path = str(annotated[0])
print('Annotated video:', vid_path)

cap = cv2.VideoCapture(vid_path)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f'Frames: {n_frames}, FPS: {fps:.1f}, duration: {n_frames/fps:.1f}s')

samples_idx = np.linspace(0, n_frames - 1, 5, dtype=int)
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, idx in zip(axes, samples_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ok, frame = cap.read()
    if not ok:
        ax.set_title(f'klatka {idx} — brak')
        ax.axis('off')
        continue
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f'klatka {idx}')
    ax.axis('off')
cap.release()
plt.tight_layout()
plt.savefig(out_dir / 'sample_keypoints_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print('Zapisalem podglad do', out_dir / 'sample_keypoints_grid.png')

## 4. Notatka decyzyjna (wpis do `../GATE.md`)

Po obejrzeniu `outputs/<sample>_labeled.mp4` w QuickTime / VLC oraz siatki `sample_keypoints_grid.png` odpowiedz na pytanie:

**Czy keypoints lądują wzrokowo sensownie na koniu (kopyta, stawy, oczy, uszy) na ≥ 80% klatek?**

TAK → punkt 2 GATE = TAK. Idź do notebooka `01_read_my_ears_replicate.ipynb`.

NIE → punkt 2 GATE = NIE. Sprawdź czy sample ma horyzontalną kamerę i jeden koń w kadrze. Jeśli mimo to keypoints się rozjeżdżają — to znaczący sygnał o domain shift; wpis do GATE.md i decyzja czy próbować notebooka 99 (Colab) czy iść z NO-GO.